In [19]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

In [8]:
# 1. GENERATE STORES (The Spoke)
retailers = ['Jewel-Osco', 'Mariano\'s', 'Target']
locations = ['Naperville', 'Lincoln Park', 'Evanston', 'Aurora', 'Oak Park', 'Schaumburg']

stores_data = []
num_stores = 50 #number of stores to consider in study
for i in range(num_stores):
    brand = np.random.choice(retailers)
    loc = np.random.choice(locations)
    # Estimate distance from Aurora, IL hub (usually 5 to 45 miles)
    dist = np.random.uniform(5, 45) 
    stores_data.append([f"STR_{i:04d}", f"{brand} - {loc} #{i}", dist])

df_stores = pd.DataFrame(stores_data, columns=['store_id', 'store_name', 'miles_from_hub'])
print (df_stores)

    store_id                     store_name  miles_from_hub
0   STR_0000        Mariano's - Evanston #0       10.850033
1   STR_0001        Mariano's - Evanston #1       26.048335
2   STR_0002     Jewel-Osco - Naperville #2        5.160089
3   STR_0003      Mariano's - Schaumburg #3       32.252732
4   STR_0004       Jewel-Osco - Evanston #4       32.629109
5   STR_0005     Jewel-Osco - Schaumburg #5       14.066743
6   STR_0006         Target - Schaumburg #6       33.837682
7   STR_0007        Mariano's - Evanston #7       42.283470
8   STR_0008             Target - Aurora #8       40.291556
9   STR_0009          Mariano's - Aurora #9       18.829427
10  STR_0010      Jewel-Osco - Evanston #10       11.463784
11  STR_0011      Jewel-Osco - Oak Park #11       27.580392
12  STR_0012      Jewel-Osco - Oak Park #12       28.609876
13  STR_0013            Target - Aurora #13       43.833142
14  STR_0014    Jewel-Osco - Schaumburg #14       27.078746
15  STR_0015  Jewel-Osco - Lincoln Park 

In [15]:
# 2. GENERATE ASSETS (The Laundry Jugs)
# 2026 ESG Specs: Every jug has a 'birth weight' and material type
assets_data = []
num_of_jugs = 100000 #update to 5,000 and eventually 1,000,000 for pilot study
for i in range(num_of_jugs):
    assets_data.append([
        f"JUG_{i:05d}", 
        "HDPE_Plastic", 
        120, # grams of plastic
        "2025-11-01" # Launch date
    ])

df_assets = pd.DataFrame(assets_data, columns=['asset_id', 'material', 'weight_grams', 'initial_date'])

#print (df_assets) 


In [16]:
# --- PHASE 3: GENERATE LOGISTICS EVENTS ---
events = []
simulation_start_date = datetime(2026, 1, 1)

for jug_id in df_assets['asset_id']:
    # 1. Reset the "clock" for EACH individual jug so they don't stack to the year 9999
    # We stagger the initial purchase over the first 60 days of the year
    current_time = simulation_start_date + timedelta(days=np.random.randint(0, 60))
    
    # 2. Simulate 1 to 10 rotations (some jugs might be newer than others)
    rotations = np.random.randint(1, 11)
    
    for r in range(rotations):
        # Pick a random store for this specific rotation
        store = df_stores.sample(1).iloc[0]
        
        # --- PHASE A: AT RETAIL (Shelf Time) ---
        shelf_life = np.random.randint(2, 14) 
        current_time += timedelta(days=shelf_life)
        events.append([jug_id, store['store_id'], 'purchased', current_time])
        
        # --- PHASE B: AT HOME (Usage) ---
        usage_life = np.random.randint(45, 91) 
        current_time += timedelta(days=usage_life)
        
        # --- PHASE C: THE RETURN LAG (Empty in garage/kitchen) ---
        return_lag = np.random.randint(3, 15)
        current_time += timedelta(days=return_lag)
        
        # Determine if the jug successfully returns or is lost
        if np.random.random() < 0.90: # 90% Return Rate
            events.append([jug_id, store['store_id'], 'returned_to_store', current_time])
            
            # --- PHASE D: BACK TO HUB (Transit + Washing) ---
            processing_time = np.random.randint(2, 7)
            current_time += timedelta(days=processing_time)
            events.append([jug_id, 'HUB_AURORA', 'washed', current_time])
            
            # The jug is now clean and ready for the next rotation 'r'
        else:
            # Jug is lost! Record the loss and EXIT the rotation loop for this jug
            events.append([jug_id, 'UNKNOWN', 'lost_to_leakage', current_time])
            break 

# Convert to DataFrame
df_events = pd.DataFrame(events, columns=['asset_id', 'location_id', 'event_type', 'timestamp'])

In [18]:
# SAVE DATA and update tables in PostgresSQL
from sqlalchemy import create_engine, text
import os

# --- PART 1: CSV BACKUP ---
output_dir = '../data'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

df_stores.to_csv(os.path.join(output_dir, 'stores.csv'), index=False)
df_assets.to_csv(os.path.join(output_dir, 'assets.csv'), index=False)
df_events.to_csv(os.path.join(output_dir, 'logistics_events.csv'), index=False)

# --- PART 2: POSTGRES REFRESH ---
import io

def postgres_upsert(df, table_name, engine):
    # Create a buffer
    output = StringIO()
    # Ensure we don't send the index, and we use a tab separator
    df.to_csv(output, sep='\t', header=False, index=False)
    output.seek(0)
    
    raw_conn = engine.raw_connection()
    cursor = raw_conn.cursor()
    try:
        # GET COLUMN NAMES FROM DATAFRAME
        columns = [f'"{col}"' for col in df.columns]
        
        # SPECIFY COLUMNS IN THE COPY COMMAND
        cursor.copy_from(output, table_name, sep='\t', null='', columns=columns)
        
        raw_conn.commit()
        print(f"Successfully uploaded {len(df)} rows.")
    except Exception as e:
        raw_conn.rollback()
        print(f"Error: {e}")
    finally:
        cursor.close()
        raw_conn.close()